# Phase 1 — Pretrained ResNet50 + Exact k-NN Baseline
This notebook is a thin wrapper: all real logic lives in `src/` and `scripts/`.
It assumes the project folder has already been uploaded to Colab (e.g. via
Google Drive or a direct upload) **with the dataset already placed** at:
- `data/images/<id>.jpg`
- `data/styles.csv`

See the repo's `README.md` for dataset setup instructions.


In [ ]:
# If running from Google Drive, mount it and cd into the project folder.
# Skip this cell if you uploaded the project directly into the Colab session.
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/fashion_recommender_project"  # <-- adjust to your path
%cd {PROJECT_DIR}


In [ ]:
!pip install -q -r requirements.txt


## 1. Verify the dataset is in place

In [ ]:
import sys
sys.path.append(".")
from src import config, data_loader

config.ensure_directories()
data_loader.check_dataset_present()
print("Dataset OK:", config.STYLES_CSV_PATH, "|", config.IMAGES_DIR)


## 2. Clean metadata, create stratified 70/15/15 split (seed 42)

In [ ]:
from src import preprocess

df_raw = data_loader.load_raw_metadata()
df_clean, clean_report = preprocess.clean_metadata(df_raw)
print("Cleaning report:", clean_report)

df_final, small_cat_report = preprocess.drop_small_categories(df_clean)
print("Small-category report:", small_cat_report)

config.METADATA_DIR.mkdir(parents=True, exist_ok=True)
df_final.to_csv(config.METADATA_CLEAN_PATH, index=False)

train_df, val_df, test_df = preprocess.stratified_split(df_final)
preprocess.save_splits(train_df, val_df, test_df)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")


## 3. Category distribution

In [ ]:
from src import visualization
visualization.plot_category_distribution(df_final, save_path=config.FIGURES_DIR / "category_distribution_top20.png")


## 4. Extract pretrained ResNet50 embeddings (train / val / test)

In [ ]:
from src import embeddings

config.set_global_seed()
model = embeddings.build_resnet50_backbone(freeze=True)

for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    loader = data_loader.build_dataloader(df, shuffle=False)
    emb, ids, cats, elapsed, n_images = embeddings.extract_embeddings(model, loader, split_name)
    embeddings.save_embeddings(split_name, emb, ids, cats, tag="baseline")


## 5. Exact k-NN retrieval + Phase 1 evaluation (Protocol A)

In [ ]:
from src import evaluation, retrieval

emb_train, ids_train, cat_train = embeddings.load_embeddings("train", tag="baseline")
emb_test, ids_test, cat_test = embeddings.load_embeddings("test", tag="baseline")

knn_index, build_time = retrieval.build_knn_index(emb_train, k_max=config.K_MAX)
neighbor_ids, neighbor_sims, query_latencies = retrieval.retrieve_top_k(
    emb_test, ids_test, ids_train, k=config.K_MAX, index=knn_index
)

gallery_lookup = evaluation.build_category_lookup(ids_train, cat_train)
pr_results = evaluation.precision_recall_at_k(neighbor_ids, gallery_lookup, cat_train, cat_test, config.K_VALUES)
map_results = {k: evaluation.mean_average_precision_at_k(neighbor_ids, gallery_lookup, cat_test, k) for k in config.MAP_K_VALUES}
cat_consistency = {k: evaluation.category_consistency_at_k(neighbor_ids, gallery_lookup, cat_test, k) for k in config.CATEGORY_CONSISTENCY_K_VALUES}

for k in config.K_VALUES:
    print(f"Precision@{k}: {pr_results[k]['precision']:.4f}  Recall@{k}: {pr_results[k]['recall']:.4f}")
for k, v in map_results.items():
    print(f"mAP@{k}: {v:.4f}")
for k, v in cat_consistency.items():
    print(f"Category Consistency@{k}: {v:.4f}")


## 6. Latency, metrics CSV, and plots

In [ ]:
import numpy as np, pandas as pd

mean_q_ms = float(np.mean(query_latencies) * 1000)
median_q_ms = float(np.median(query_latencies) * 1000)
p95_q_ms = float(np.percentile(query_latencies, 95) * 1000)

extra_rows = {
    "num_test_queries": len(ids_test),
    "num_train_gallery_items": len(ids_train),
    "knn_index_build_time_s": build_time,
    "query_latency_mean_ms": mean_q_ms,
    "query_latency_median_ms": median_q_ms,
    "query_latency_p95_ms": p95_q_ms,
}
metrics_df = evaluation.metrics_to_dataframe(pr_results, map_results, cat_consistency, extra_rows)
metrics_df.to_csv(config.METRICS_CSV_PATH, index=False)
print(f"Saved: {config.METRICS_CSV_PATH}")

pd.DataFrame([
    {"stage": "knn_index_build_s", "value": build_time},
    {"stage": "query_latency_mean_ms", "value": mean_q_ms},
    {"stage": "query_latency_median_ms", "value": median_q_ms},
    {"stage": "query_latency_p95_ms", "value": p95_q_ms},
]).to_csv(config.LATENCY_LOG_PATH, index=False)
print(f"Saved: {config.LATENCY_LOG_PATH}")

visualization.plot_precision_recall_and_consistency(
    config.K_VALUES, pr_results, cat_consistency,
    title_suffix="(pretrained ResNet50, exact k-NN)",
    save_path=config.FIGURES_DIR / "phase1_precision_recall_category_consistency.png",
)
metrics_df


## 7. Qualitative examples

In [ ]:
example_indices = np.random.RandomState(config.SEED).choice(len(ids_test), size=3, replace=False)
for idx in example_indices:
    idx = int(idx)
    row = test_df[test_df["id"] == ids_test[idx]].iloc[0]
    save_path = config.QUALITATIVE_DIR / f"query_{ids_test[idx]}.png"
    visualization.visualize_query(
        ids_test[idx], cat_test[idx], row["image_path"],
        neighbor_ids[idx], neighbor_sims[idx], train_df, k=5, save_path=save_path,
    )


## Phase 1 complete
Check `results/metrics/metrics_baseline_exact_knn.csv`, `results/logs/latency_log.csv`, and
`results/figures/` for everything generated above. Once you're satisfied with these numbers,
move on to `notebooks/phase2_improvements.ipynb`.
